In [1]:
import pandas as pd
from keras.src.callbacks import early_stopping
from keras.src.utils.module_utils import tensorflow
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder, OneHotEncoder
import pickle


In [2]:
df= pd.read_csv("Churn_Modelling.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  str    
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  str    
 5   Gender           10000 non-null  str    
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), str(3)
memory usage: 1.2 MB


In [4]:
df["Surname"].value_counts()

Surname
Smith        32
Scott        29
Martin       29
Walker       28
Brown        26
             ..
Salinas       1
Cleveland     1
Kashiwagi     1
Aldridge      1
Burbidge      1
Name: count, Length: 2932, dtype: int64

In [5]:
df.drop(columns=["RowNumber","CustomerId","Surname"],inplace=True)

In [6]:
# Encoding categorical vairables

encoder=LabelEncoder()
df["Gender"]=encoder.fit_transform(df["Gender"])

In [7]:
ohe=OneHotEncoder(sparse_output=False)
geo_encoded=ohe.fit_transform(df[["Geography"]])
geo_encoded

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [8]:
geo_encoded_df=pd.DataFrame(
    geo_encoded,
    columns=ohe.get_feature_names_out(["Geography"])

)
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [9]:
#Combine

df.drop(columns=["Geography"], inplace=True)

df = pd.concat([df, geo_encoded_df], axis=1)

In [10]:
df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


In [11]:
with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(encoder, file)

with open("label_encoder_geography.pkl", "wb") as file:
    pickle.dump(ohe, file)


In [12]:
X=df.drop("Exited",axis=1)
y=df["Exited"]

In [13]:
#train test split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scalar=StandardScaler()
X_train=scalar.fit_transform(X_train)
X_test=scalar.transform(X_test)

In [14]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scalar, file)

# ANN Implementation

In [16]:
import tensorflow as tf
from tensorflow.keras.models import  Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

## Make model

In [19]:
from tensorflow.keras import Input

model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

In [20]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

# Compile the model

In [21]:
opt= tf.keras.optimizers.Adam(learning_rate=0.001) #Do this to set ur desired learning rate

In [22]:
model.compile(loss="binary_crossentropy", optimizer=opt, metrics=["accuracy"])

## Setup tensorboard

In [23]:
log_dir= "logs_fit"+ datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [26]:
tensorboard_callback= TensorBoard(log_dir=log_dir,histogram_freq=1)

## Setup Early stopping

In [27]:
early_stopping_callback=EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

## Training the model

In [34]:
history=model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=150,
    callbacks=[tensorboard_callback,early_stopping_callback]
)

Epoch 1/150
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8664 - loss: 0.3161 - val_accuracy: 0.8565 - val_loss: 0.3402
Epoch 2/150
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8675 - loss: 0.3164 - val_accuracy: 0.8575 - val_loss: 0.3369
Epoch 3/150
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8683 - loss: 0.3142 - val_accuracy: 0.8600 - val_loss: 0.3391
Epoch 4/150
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8706 - loss: 0.3116 - val_accuracy: 0.8640 - val_loss: 0.3360
Epoch 5/150
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8696 - loss: 0.3119 - val_accuracy: 0.8595 - val_loss: 0.3416
Epoch 6/150
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8686 - loss: 0.3107 - val_accuracy: 0.8600 - val_loss: 0.3401
Epoch 7/150
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8704 - loss: 0.3091 - val_accuracy: 0.8620 - val_loss: 0.3416
Epoch 8/150
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8704 - loss: 0.3081 - val_accu

In [35]:
model.save("model.h5")

In [36]:
# Launch tensorboard extension

%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [38]:
%tensorboard --logdir logs_fit20260905-163423